In [ ]:
# Label Studio Crop Merger - Jupyter Notebook Version
# Run each cell sequentially to merge annotated crops back into full images

# Cell 1: Import libraries and setup
import os
import json
import requests
from PIL import Image, ImageDraw, ImageFont
import numpy as np
from pathlib import Path
from label_studio_sdk import Client
from urllib.parse import urlparse
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output
import re
from collections import defaultdict

# Enable inline plotting
%matplotlib inline

# Configuration
LS_URL = 'http://129.97.250.147:8080/'
API_TOKEN = 'ebdc6fa5f2c3abcd502b55d5ccc1dc0e4ae9f68d'
PROJECT_ID = 95

print("✓ Libraries imported and configuration set")

# Cell 2: Initialize the CropMerger class
class CropMerger:
    def __init__(self, ls_url, api_token, project_id):
        self.client = Client(url=ls_url, api_key=api_token)
        self.project_id = project_id
        self.project = self.client.get_project(project_id)
        print(f"✓ Connected to Label Studio project {project_id}")
        
    def get_all_tasks(self):
        """Get all tasks from the project"""
        try:
            tasks = self.project.get_tasks()
            return tasks
        except Exception as e:
            print(f"❌ Error fetching tasks: {str(e)}")
            return []
    
    def download_image(self, image_url, filename):
        """Download image from Label Studio"""
        try:
            if image_url.startswith('/'):
                full_url = f"{LS_URL.rstrip('/')}{image_url}"
            else:
                full_url = image_url
            
            headers = {'Authorization': f'Token {API_TOKEN}'}
            response = requests.get(full_url, headers=headers)
            response.raise_for_status()
            
            temp_path = f"temp_{filename}"
            with open(temp_path, 'wb') as f:
                f.write(response.content)
            
            return temp_path
        except Exception as e:
            print(f"❌ Error downloading image {filename}: {str(e)}")
            return None
    
    def parse_crop_filename(self, filename):
        """Parse crop filename to extract base name and crop number"""
        # Pattern: basename_crop_001.jpg
        pattern = r'(.+)_crop_(\d+)\.(jpg|jpeg|png|bmp)$'
        match = re.match(pattern, filename, re.IGNORECASE)
        
        if match:
            base_name = match.group(1)
            crop_num = int(match.group(2))
            extension = match.group(3)
            return base_name, crop_num, extension
        return None, None, None
    
    def group_crops_by_base_image(self, tasks):
        """Group crop tasks by their original base image"""
        crop_groups = defaultdict(list)
        
        for task in tasks:
            task_data = task.get('data', {})
            image_path_key = 'image' if 'image' in task_data else 'img'
            image_url = task_data.get(image_path_key, '')
            
            if not image_url:
                continue
                
            filename = Path(urlparse(image_url).path).name
            filename = filename.split('-')[-1]  # Handle full URL paths
            base_name, crop_num, extension = self.parse_crop_filename(filename)
            
            if base_name and crop_num:
                crop_groups[base_name].append({
                    'task': task,
                    'crop_num': crop_num,
                    'filename': filename,
                    'image_url': image_url
                })
        
        # Sort crops by crop number for each base image
        for base_name in crop_groups:
            crop_groups[base_name].sort(key=lambda x: x['crop_num'])
        
        return crop_groups
    
    def calculate_grid_dimensions(self, num_crops):
        """Calculate the most likely grid dimensions based on number of crops"""
        # Common grid patterns
        common_grids = [
            (1, 2), (1, 3), (1, 4), (1, 5), (1, 6),
            (2, 2), (2, 3), (2, 4), (2, 5), (2, 6),
            (3, 2), (3, 3), (3, 4), (3, 5), (3, 6),
            (4, 2), (4, 3), (4, 4), (4, 5), (4, 6),
            (5, 2), (5, 3), (5, 4), (5, 5), (5, 6),
            (6, 6)
        ]
        
        for v_blocks, h_blocks in common_grids:
            if v_blocks * h_blocks == num_crops:
                return v_blocks, h_blocks
        
        # If no common pattern, try to find factors
        for i in range(1, int(num_crops**0.5) + 1):
            if num_crops % i == 0:
                return i, num_crops // i
        
        # Fallback: single row
        return 1, num_crops
    
    def transform_annotations_to_full_image(self, annotations, crop_position, crop_size, full_image_size):
        """Transform crop annotations back to full image coordinates"""
        crop_x, crop_y = crop_position
        crop_width, crop_height = crop_size
        full_width, full_height = full_image_size
        
        transformed_annotations = []
        
        for annotation in annotations:
            for result in annotation.get('result', []):
                if result.get('type') == 'keypointlabels':
                    value = result.get('value', {})
                    
                    # Get crop coordinates (percentages)
                    crop_x_percent = value.get('x', 0)
                    crop_y_percent = value.get('y', 0)
                    
                    # Convert to crop pixel coordinates
                    crop_x_pixel = (crop_x_percent / 100) * crop_width
                    crop_y_pixel = (crop_y_percent / 100) * crop_height
                    
                    # Transform to full image pixel coordinates
                    full_x_pixel = crop_x + crop_x_pixel
                    full_y_pixel = crop_y + crop_y_pixel
                    
                    # Convert back to percentages for full image
                    full_x_percent = (full_x_pixel / full_width) * 100
                    full_y_percent = (full_y_pixel / full_height) * 100
                    
                    # Create new result with transformed coordinates
                    new_result = result.copy()
                    new_result['value'] = value.copy()
                    new_result['value']['x'] = full_x_percent
                    new_result['value']['y'] = full_y_percent
                    new_result['to_name'] = 'image'
                    
                    transformed_annotations.append(new_result)
        
        return transformed_annotations
    
    def upload_merged_image_to_labelstudio(self, merged_image_path, base_name, all_annotations):
        """Upload merged image back to Label Studio as new task"""
        try:
            merged_filename = Path(merged_image_path).name
            
            # Step 1: Upload the merged image file
            with open(merged_image_path, 'rb') as f:
                upload_response = self.client.make_request(
                    'POST',
                    '/api/projects/{}/import'.format(self.project_id),
                    files={'file': (merged_filename, f, 'image/jpeg')}
                )
            
            if upload_response.status_code != 201:
                print(f"❌ Failed to upload merged image: {upload_response.status_code}")
                return None
            
            # Step 2: Create task with annotations in single request
            task_payload = {
                'data': {'image': f'/data/upload/{self.project_id}/{merged_filename}'}
            }
            
            # Add annotations to the task payload if any exist
            if all_annotations:
                task_payload['annotations'] = [{
                    'result': all_annotations,
                    'was_cancelled': False,
                    'ground_truth': False,
                    'created_username': 'crop_merger',
                    'updated_username': 'crop_merger',
                }]
            
            # Single API call to create task with annotations
            create_response = self.client.make_request(
                'POST',
                f'/api/projects/{self.project_id}/tasks/',
                json=task_payload
            )
            
            if create_response.status_code != 201:
                print(f"❌ Failed to create Label Studio task: {create_response.status_code}")
                print(f"Response: {create_response.text}")
                return None
            
            new_task = create_response.json()
            new_task_id = new_task['id']
            
            print(f"🎯 Uploaded to Label Studio: Task ID {new_task_id}")
            return new_task_id
            
        except Exception as e:
            print(f"❌ Error uploading to Label Studio: {str(e)}")
            return None

    def merge_crops_to_full_image(self, base_name, crop_data, v_blocks=None, h_blocks=None, upload_to_ls=True):
        """Merge crops back into full image with annotations"""
        print(f"🔄 Processing {base_name} with {len(crop_data)} crops...")
        
        if not crop_data:
            print("❌ No crop data provided")
            return None
        
        # Auto-detect grid dimensions if not provided
        if v_blocks is None or h_blocks is None:
            v_blocks, h_blocks = self.calculate_grid_dimensions(len(crop_data))
            print(f"📐 Auto-detected grid: {v_blocks} × {h_blocks}")
        
        # Download first crop to get dimensions
        first_crop = crop_data[0]
        temp_crop_path = self.download_image(first_crop['image_url'], first_crop['filename'])
        if not temp_crop_path:
            return None
        
        try:
            crop_img = Image.open(temp_crop_path)
            crop_width, crop_height = crop_img.size
            
            # Calculate full image dimensions
            full_width = crop_width * h_blocks
            full_height = crop_height * v_blocks
            
            print(f"📊 Crop size: {crop_width} × {crop_height}")
            print(f"📊 Full image size: {full_width} × {full_height}")
            
            # Create full image canvas
            full_image = Image.new('RGB', (full_width, full_height), color='white')
            all_annotations = []
            
            # Process each crop
            for crop_info in crop_data:
                crop_num = crop_info['crop_num']
                
                # Calculate grid position (1-based to 0-based conversion)
                grid_index = crop_num - 1
                grid_row = grid_index // h_blocks
                grid_col = grid_index % h_blocks
                
                # Calculate pixel position in full image
                x_pos = grid_col * crop_width
                y_pos = grid_row * crop_height
                
                print(f"📍 Crop {crop_num}: grid({grid_row}, {grid_col}) → pixel({x_pos}, {y_pos})")
                
                # Download and place crop image
                if crop_info['filename'] != first_crop['filename']:
                    temp_crop_path = self.download_image(crop_info['image_url'], crop_info['filename'])
                    if temp_crop_path:
                        crop_img = Image.open(temp_crop_path)
                        os.remove(temp_crop_path)
                    else:
                        continue
                
                full_image.paste(crop_img, (x_pos, y_pos))
                
                # Transform annotations
                task_annotations = crop_info['task'].get('annotations', [])
                if task_annotations:
                    transformed = self.transform_annotations_to_full_image(
                        task_annotations,
                        (x_pos, y_pos),
                        (crop_width, crop_height),
                        (full_width, full_height)
                    )
                    all_annotations.extend(transformed)
                    print(f"  📌 Added {len(transformed)} annotations from crop {crop_num}")
            
            # Save merged image locally
            output_dir = "merged_images"
            os.makedirs(output_dir, exist_ok=True)
            
            # Determine file extension from first crop
            extension = Path(first_crop['filename']).suffix
            merged_filename = f"{base_name}_merged{extension}"
            merged_path = os.path.join(output_dir, merged_filename)
            
            full_image.save(merged_path)
            print(f"💾 Saved merged image: {merged_path}")
            
            # Create annotation file (local backup)
            annotation_path = None
            if all_annotations:
                merged_annotations = {
                    'data': {'image': merged_filename},
                    'annotations': [{
                        'result': all_annotations,
                        'was_cancelled': False,
                        'ground_truth': False,
                    }]
                }
                
                annotation_path = os.path.join(output_dir, f"{base_name}_annotations.json")
                with open(annotation_path, 'w') as f:
                    json.dump(merged_annotations, f, indent=2)
                print(f"📝 Saved annotations: {annotation_path}")
            
            # Upload to Label Studio
            ls_task_id = None
            if upload_to_ls:
                print(f"📤 Uploading {merged_filename} to Label Studio...")
                ls_task_id = self.upload_merged_image_to_labelstudio(merged_path, base_name, all_annotations)
            
            return {
                'base_name': base_name,
                'merged_image_path': merged_path,
                'annotation_path': annotation_path,
                'total_annotations': len(all_annotations),
                'grid_size': (v_blocks, h_blocks),
                'image_size': (full_width, full_height),
                'labelstudio_task_id': ls_task_id
            }
        
        finally:
            if os.path.exists(temp_crop_path):
                os.remove(temp_crop_path)
    
    def preview_merge_plan(self):
        """Preview what will be merged"""
        tasks = self.get_all_tasks()
        crop_groups = self.group_crops_by_base_image(tasks)
        
        if not crop_groups:
            print("❌ No crop tasks found in project")
            return None
        
        print("🔍 MERGE PREVIEW")
        print("=" * 60)
        
        merge_plan = {}
        for base_name, crops in crop_groups.items():
            v_blocks, h_blocks = self.calculate_grid_dimensions(len(crops))
            
            total_annotations = 0
            for crop in crops:
                task_annotations = crop['task'].get('annotations', [])
                for annotation in task_annotations:
                    total_annotations += len(annotation.get('result', []))
            
            merge_plan[base_name] = {
                'crops': crops,
                'grid_size': (v_blocks, h_blocks),
                'total_annotations': total_annotations
            }
            
            print(f"📁 {base_name}")
            print(f"   • {len(crops)} crops in {v_blocks}×{h_blocks} grid")
            print(f"   • {total_annotations} total annotations")
            print(f"   • Crop numbers: {[c['crop_num'] for c in crops]}")
            print()
        
        return merge_plan
    
    def merge_all_crop_groups(self, merge_plan=None, upload_to_ls=True):
        """Merge all crop groups back to full images"""
        if merge_plan is None:
            merge_plan = self.preview_merge_plan()
        
        if not merge_plan:
            return []
        
        results = []
        total_groups = len(merge_plan)
        
        print(f"🚀 Starting merge process for {total_groups} image groups...")
        if upload_to_ls:
            print(f"📤 Will upload merged images back to Label Studio project {self.project_id}")
        print("=" * 60)
        
        # Create progress bar
        progress = widgets.IntProgress(
            value=0,
            min=0,
            max=total_groups,
            description='Merging:',
            bar_style='info',
            style={'bar_color': 'green'},
            orientation='horizontal'
        )
        display(progress)
        
        for i, (base_name, plan_data) in enumerate(merge_plan.items(), 1):
            progress.value = i
            
            crops = plan_data['crops']
            v_blocks, h_blocks = plan_data['grid_size']
            
            result = self.merge_crops_to_full_image(base_name, crops, v_blocks, h_blocks, upload_to_ls)
            if result:
                results.append(result)
            
            print(f"✅ Completed {i}/{total_groups}: {base_name}")
            print("-" * 40)
        
        progress.bar_style = 'success'
        
        print(f"\n🎉 Merge process completed!")
        print(f"✅ Successfully merged {len(results)} images")
        print(f"📁 Local files saved to: ./merged_images/")
        if upload_to_ls:
            uploaded_count = sum(1 for r in results if r.get('labelstudio_task_id'))
            print(f"📤 Uploaded {uploaded_count} images to Label Studio")
        
        return results

# Initialize the merger
merger = CropMerger(LS_URL, API_TOKEN, PROJECT_ID)

# Cell 3: Create interactive controls
print("🎮 Crop Merger Controls:")

preview_button = widgets.Button(
    description='Preview Merge Plan',
    button_style='info',
    icon='search'
)

merge_local_button = widgets.Button(
    description='Merge (Local Only)',
    button_style='warning',
    icon='download'
)

merge_upload_button = widgets.Button(
    description='Merge & Upload to LS',
    button_style='success',
    icon='cloud-upload'
)

upload_toggle = widgets.Checkbox(
    value=True,
    description='Auto-upload to Label Studio',
    style={'description_width': 'initial'}
)

output_area = widgets.Output()

# Display widgets
display(widgets.VBox([
    widgets.HBox([preview_button, merge_local_button, merge_upload_button]),
    upload_toggle,
    output_area
]))

# Cell 4: Define event handlers
merge_plan_data = None

def on_preview_clicked(b):
    global merge_plan_data
    with output_area:
        clear_output()
        print("🔍 Analyzing crop tasks...")
        merge_plan_data = merger.preview_merge_plan()

def on_merge_local_clicked(b):
    global merge_plan_data
    with output_area:
        clear_output()
        if merge_plan_data is None:
            print("🔍 First analyzing crop tasks...")
            merge_plan_data = merger.preview_merge_plan()
        
        if merge_plan_data:
            print(f"🚀 Starting local merge process...")
            results = merger.merge_all_crop_groups(merge_plan_data, upload_to_ls=False)
            display_merge_summary(results, uploaded=False)
        else:
            print("❌ No crop groups found to merge")

def on_merge_upload_clicked(b):
    global merge_plan_data
    with output_area:
        clear_output()
        if merge_plan_data is None:
            print("🔍 First analyzing crop tasks...")
            merge_planfi_data = merger.preview_merge_plan()
        
        if merge_plan_data:
            print(f"🚀 Starting merge and upload process...")
            results = merger.merge_all_crop_groups(merge_plan_data, upload_to_ls=True)
            display_merge_summary(results, uploaded=True)
        else:
            print("❌ No crop groups found to merge")

def display_merge_summary(results, uploaded=False):
    """Display detailed merge summary"""
    if results:
        print("\n" + "="*60)
        print("📈 MERGE SUMMARY")
        print("="*60)
        
        total_annotations = sum(r['total_annotations'] for r in results)
        uploaded_count = sum(1 for r in results if r.get('labelstudio_task_id')) if uploaded else 0
        
        for result in results:
            print(f"✅ {result['base_name']}")
            print(f"   • Grid: {result['grid_size'][0]}×{result['grid_size'][1]}")
            print(f"   • Size: {result['image_size'][0]}×{result['image_size'][1]} pixels")
            print(f"   • Annotations: {result['total_annotations']}")
            print(f"   • Local file: {result['merged_image_path']}")
            if result.get('annotation_path'):
                print(f"   • Annotations JSON: {result['annotation_path']}")
            if result.get('labelstudio_task_id'):
                print(f"   • Label Studio Task ID: {result['labelstudio_task_id']}")
            print()
        
        print(f"🎯 Total images merged: {len(results)}")
        print(f"📌 Total annotations preserved: {total_annotations}")
        print(f"📁 Local files saved to: ./merged_images/")
        
        if uploaded:
            print(f"📤 Uploaded to Label Studio: {uploaded_count}/{len(results)} images")
            if uploaded_count > 0:
                print(f"🎯 Check Label Studio project {PROJECT_ID} for new merged tasks!")
        
        print("🎉 All merges completed successfully!")

# Attach event handlers
preview_button.on_click(on_preview_clicked)
merge_local_button.on_click(on_merge_local_clicked)
merge_upload_button.on_click(on_merge_upload_clicked)

print("🎮 Controls ready!")
print("💡 Options:")
print("   • 'Preview Merge Plan' - See what will be merged")
print("   • 'Merge (Local Only)' - Create merged files locally only")
print("   • 'Merge & Upload to LS' - Create files AND upload to Label Studio")
print(f"🎯 Target Label Studio project: {PROJECT_ID}")

✓ Libraries imported and configuration set
✓ Connected to Label Studio project 95
🎮 Crop Merger Controls:


🎮 Controls ready!
💡 Options:
   • 'Preview Merge Plan' - See what will be merged
   • 'Merge (Local Only)' - Create merged files locally only
   • 'Merge & Upload to LS' - Create files AND upload to Label Studio
🎯 Target Label Studio project: 95
